In [1]:
import pyaudio
import wave
import keyboard
import time

FORMAT = pyaudio.paInt16
CHANNELS = 1
RATE = 44100
CHUNK = 1024
OUTPUT_FILENAME = "recording.wav"

In [3]:
audio = pyaudio.PyAudio()

    # Start Recording
stream = audio.open(format=FORMAT, channels=CHANNELS,
                        rate=RATE, input=True,
                        frames_per_buffer=CHUNK)
print("Recording... Press 's' to stop.")
frames = []

while True:
    data = stream.read(CHUNK)
    frames.append(data)
    if keyboard.is_pressed('s'):  # If 's' is pressed, stop recording
        print("Stopped recording.")
        break
    # Stop Recording
stream.stop_stream()
stream.close()
audio.terminate()

    # Save the recorded data as a WAV file
with wave.open(OUTPUT_FILENAME, 'wb') as wf:
    wf.setnchannels(CHANNELS)
    wf.setsampwidth(audio.get_sample_size(FORMAT))
    wf.setframerate(RATE)
    wf.writeframes(b''.join(frames))

print(f"Recording saved as {OUTPUT_FILENAME}")


Recording... Press 's' to stop.
Stopped recording.
Recording saved as recording.wav


In [4]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install torchaudio

Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install transformers

Note: you may need to restart the kernel to use updated packages.


In [7]:
from transformers import Wav2Vec2ForSequenceClassification, AutoFeatureExtractor
import torch, torchaudio

model_id = "facebook/mms-lid-4017"

processor = AutoFeatureExtractor.from_pretrained(model_id)
model = Wav2Vec2ForSequenceClassification.from_pretrained(model_id)

c:\Users\nadun\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
%pip install soundfile

Note: you may need to restart the kernel to use updated packages.


In [9]:
import soundfile as sf
import torch

speech, sr = sf.read("recording.wav", dtype="float32")
speech = torch.from_numpy(speech).T  # convert numpy → torch [channels, time]

#if sampling rate is not 16kHz, resample it
if sr != 16000:
    resampler = torchaudio.transforms.Resample(sr, 16000)
    speech = resampler(speech)

inputs = processor(speech, sampling_rate=16000, return_tensors="pt")

with torch.no_grad():
    logits = model(**inputs).logits

lang_id = torch.argmax(logits, dim=-1).item()
detected_lang = model.config.id2label[lang_id]

C:\Users\nadun\AppData\Local\Temp\ipykernel_30180\1406078612.py:5: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4424.)
  speech = torch.from_numpy(speech).T  # convert numpy → torch [channels, time]


In [10]:
print(detected_lang)

mpm
